# tenuretrack

**Nobody tells assistant professors the numbers. This notebook computes them.**

Give it your ORCID, your university, and the year your appointment began. It
builds a cohort of early-career faculty in your subfield from
[OpenAlex](https://openalex.org) and shows where your record sits at the same
point on the tenure clock.

No install, no Python. Run the cells in order from the top.

### What you need

1. Your ORCID, free at [orcid.org](https://orcid.org).
2. Your university, and the first calendar year of your appointment.
3. Your work email. OpenAlex asks callers to identify themselves; it goes to
   OpenAlex and nowhere else.
4. **A free OpenAlex API key, before you start.** Thirty seconds at
   [openalex.org/settings/api](https://openalex.org/settings/api), no payment
   details, and it cannot be billed. **With it, 10 to 40 minutes. Without it,
   two or three days**, because an unidentified caller gets about 1,000 requests
   a day and this spends a few thousand.

An interruption costs you nothing either way. Everything fetched is cached.

### Two things to know

Building the cohort downloads records carrying other faculty's names. They stay
on this temporary machine. The report never contains a cohort member's name or
individual numbers. The one person named in your results is you.

These numbers describe what a group of people did. They are not a standard, and
nobody in the cohort agreed to be measured.

---
## Step 1: Install the tool

Once per session, about a minute. Colab gives you a fresh machine each time.

In [ ]:
#@title Install tenuretrack { display-mode: "form" }
!pip install --quiet "git+https://github.com/sp8rks/tenuretrack.git"

import tenuretrack
from tenuretrack import notebook as _nb

# This notebook and the package it drives are two files that travel apart. A
# copy saved in Drive can be newer than what pip just fetched from the main
# branch, and Colab keeps an already-imported module even after an upgrade.
# Checking the names here, by hand, turns what would otherwise be an
# ImportError three cells later into a sentence that says what to do.
NEEDED = (
    "describe_config",
    "keep_topics",
    "list_results",
    "prompt_for_api_key",
    "run_cli",
    "set_api_key",
    "set_mailto",
    "set_peer_group",
    "zip_results",
)
missing = [name for name in NEEDED if not hasattr(_nb, name)]
if missing:
    raise RuntimeError(
        "The installed tenuretrack (" + tenuretrack.__version__ + ") is older "
        "than this notebook, and is missing: " + ", ".join(missing) + ". "
        "Choose Runtime > Restart session, then run this cell again. If it "
        "still fails, this notebook is ahead of the main branch on GitHub."
    )

print("tenuretrack", tenuretrack.__version__, "is ready.")


---
## Step 2 (optional): Keep your work in Google Drive

Colab throws the machine away when you close the tab. Connecting Drive saves the
downloaded records and your results there, so a long build can resume tomorrow.
Skip this cell to keep everything on the temporary machine.

The saved cache includes cohort members' names, in a folder only you can see.

In [ ]:
#@title Connect Google Drive (optional) { display-mode: "form" }
import os
from pathlib import Path

use_drive = True  #@param {type:"boolean"}
folder_name = "tenuretrack"  #@param {type:"string"}

if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    workdir = Path("/content/drive/MyDrive") / folder_name
else:
    workdir = Path("/content") / folder_name

workdir.mkdir(parents=True, exist_ok=True)
os.chdir(workdir)
print("Working in:", workdir)

---
## Step 3: Tell it about you

Fill in the boxes and run the cell. It proposes the three topics that best
describe what you publish on, with the journals they ran in.

**Schools like mine.** `only_schools_like_mine` narrows the cohort to
institutions closest to yours in subfield output. Off compares you against your
whole subfield, which is the question most people mean. If you turn it on,
`how_many_schools` sets how many, and **15 is too few**: a cohort runs to about
two people per institution, so fifteen schools leaves a group in the tens, where
p25 and p75 swing on a single person. 50 is a reasonable start. This orders
schools by subfield output, which is not a prestige ranking; no such ranking
exists in OpenAlex to use.

**A stopped clock.** `years_the_clock_was_stopped` covers parental leave, medical
leave, or a pandemic extension. Leave it at 0 if none applies. In your sixth
calendar year with one year stopped you are at clock year five, and should be
compared against people at year five. Your papers from all six years still count:
an extension gives time back, it does not un-write what you published. The number
stays here and in your report. It is not sent to OpenAlex, and nothing about why
the clock stopped is asked for or recorded.

**The key.** The cell asks for it when you run it. Not got one?
[openalex.org/settings/api](https://openalex.org/settings/api), thirty seconds.
You can skip the key by pressing Enter and run the slower way. There is no box
for it because Colab saves box values into the notebook file, where a key would
travel with the file; to avoid retyping it, use Colab's secret store (key icon,
left sidebar, named `OPENALEX_API_KEY`).

This writes `benchmark.yaml`. Running it again after step 5 would discard your
topic choices, so it stops instead. Tick `start_over` if that is what you want.

In [ ]:
#@title Your details { display-mode: "form" }
your_email = ""  #@param {type:"string"}
orcid = ""  #@param {type:"string"}
university = ""  #@param {type:"string"}
appointment_start_year = 2019  #@param {type:"integer"}
years_the_clock_was_stopped = 0  #@param {type:"integer"}
only_schools_like_mine = False  #@param {type:"boolean"}
how_many_schools = 50  #@param {type:"integer"}
start_over = False  #@param {type:"boolean"}

# No form box for the API key on purpose: Colab writes form values back into
# the notebook file, and a secret does not belong in a file you might share.
from tenuretrack.notebook import (
    prompt_for_api_key,
    run_cli,
    set_api_key,
    set_mailto,
    set_peer_group,
)

set_mailto(your_email)
print(set_api_key(prompt_for_api_key()))
print("Looking up", orcid, "at", university)
print()

options = [
    "--orcid", orcid,
    "--institution", university,
    "--start", appointment_start_year,
    "--clock-extension", years_the_clock_was_stopped,
]
run_cli("init", *options, *(["--force"] if start_over else []))

print()
print(set_peer_group("benchmark.yaml", how_many_schools if only_schools_like_mine else 0))


---
## Step 4: Check the topics

The cohort is everyone whose research sits in these topics, so this is the choice
that matters most. The list comes from your own papers.

In [ ]:
#@title Show what the tool proposed { display-mode: "form" }
from tenuretrack.notebook import describe_config

print(describe_config("benchmark.yaml"))

---
## Step 5: Keep the topics that are yours

Type the numbers to keep, for example `1, 2`, or `all`.

Look back at "people this topic brings in" in step 3. That decides how long step 6
takes and how close the cohort is to your work, and how central a topic feels does
not predict it: a topic carrying three of your papers can bring in five times the
people of one carrying thirty.

Dropped topics are recorded as deliberately left out, so your report can say what
the cohort does not cover.

In [ ]:
#@title Choose your topics { display-mode: "form" }
keep = "all"  #@param {type:"string"}
why_dropped = "outside my subfield"  #@param {type:"string"}

from tenuretrack.notebook import describe_config, keep_topics

keep_topics("benchmark.yaml", keep, note=why_dropped)
print(describe_config("benchmark.yaml"))

---
## Step 6: Build the cohort

The long one. Leave the tab open. Progress prints as it goes, with the day's
OpenAlex allowance remaining.

If it stops, from a lost connection or a spent allowance, just run the cell again:
everything fetched is saved, so it resumes. With the key this finishes in one
sitting. Without one, expect two or three visits.

In [ ]:
#@title Build the cohort and compute the norms { display-mode: "form" }
from tenuretrack.notebook import run_cli

run_cli("run")

---
## Step 7: Read your report

`report.pdf` is the whole thing in one file, with the charts: you against the
cohort, the subfield's own numbers, how the cohort was built, and where it
publishes. The same content is in `report.md` if you would rather read text.

Citations are shown for you but not compared: your papers have had a few years to
accumulate them and your cohort's have had ten or more.

In [ ]:
#@title Show the report { display-mode: "form" }
from pathlib import Path

from IPython.display import Markdown, display

from tenuretrack.notebook import list_results

report = Path("results/report.md")
if report.exists():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print("No report yet. Run step 6 first.")

print("\nFiles written:")
for path in list_results("results"):
    print(" ", path)

---
## Step 8: Download your results

Packages the PDF, the report, the tables, and the slides into one zip. The files
are scanned for cohort members' names and IDs first; if the scan finds one,
nothing is written and the cell says what it found.

In [ ]:
#@title Download the results { display-mode: "form" }
from tenuretrack.notebook import zip_results

bundle = zip_results("results", "tenuretrack-results.zip")
print("Packaged:", bundle)

try:
    from google.colab import files

    files.download(str(bundle))
except ImportError:
    print("Not running in Colab. The zip is next to your results folder.")

---
## What this cannot answer

Teaching, mentoring, service, funding, software, datasets, and public scholarship
do not appear in OpenAlex, and they are a large part of the job. See
[docs/beyond-papers.md](https://github.com/sp8rks/tenuretrack/blob/main/docs/beyond-papers.md).

OpenAlex splits some people across profiles and merges others with namesakes. The
cohort keeps only people it can identify confidently, which tilts it toward
distinctive names. The full method is in
[docs/methods.md](https://github.com/sp8rks/tenuretrack/blob/main/docs/methods.md).

Something wrong? Open an issue at
[github.com/sp8rks/tenuretrack](https://github.com/sp8rks/tenuretrack/issues).
Please do not paste cohort names into an issue.